<a href="https://colab.research.google.com/github/s-araromi/flyrank-ml-internship-sulaimon/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-araromi/flyrank-ml-internship-sulaimon/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Field classification

**Features — all calculated using March 2026 only**

1. `march_impressions` — total measured Search Console impressions during March.
2. `march_clicks` — total measured Search Console clicks during March.
3. `march_ctr_pct` — March clicks divided by March impressions, expressed as a percentage.
4. `march_avg_position` — March average search position, weighted by impressions where possible.
5. `march_active_search_days` — number of measured March days on which the page received at least one impression.

**Label / proxy**

- `declined_next_month` — 1 when April clicks are at least 20% lower than March clicks; otherwise 0.
- The label is calculated only for eligible pages with measured Search Console data and at least one March click.

**Context**

- `client_hash_id` — used for grouping, joining and later client-separated validation.
- `content_hash_id` — used to identify and join a content item.
- `report_date` — used to assign records to the March feature window or April outcome window.
- Content descriptors from `dim_content` may be retained for interpretation but will not automatically become model features.

**Excluded**

- `click_change_pct_apr_vs_mar` — derived from the outcome and deliberately used only in the leakage demonstration.
- April clicks, impressions, CTR and position — future information at the March decision moment.
- `client_hash_id` and `content_hash_id` — pseudonymous identifiers, not predictive features.
- Rows where `gsc_data_available` is not true — unavailable measurements must not be interpreted as zero activity.

In [9]:
# Secure setup: dependencies, Hugging Face authentication, and data paths
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "duckdb>=1.0",
        "huggingface_hub>=0.24",
        "pandas>=2.2",
        "numpy>=1.26",
        "scikit-learn>=1.4",
    ],
    check=True,
)

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Read the token securely from Colab Secrets.
hf_token = userdata.get("HF_TOKEN")

if not hf_token or not hf_token.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN is missing or invalid. Check Colab Secrets and "
        "enable notebook access for HF_TOKEN."
    )

# Create an in-memory DuckDB connection.
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Pass the token as a parameter so it never appears in the notebook's SQL text.
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute(
    """
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
    """
)

del hf_token

# Warehouse paths
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DAILY_FACT = f"{WAREHOUSE}/fact_content_daily_performance"

MARCH = f"read_parquet('{DAILY_FACT}/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{DAILY_FACT}/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{WAREHOUSE}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{WAREHOUSE}/dim_clients.parquet')"

print("Secure setup complete.")
print("Feature window: March 2026")
print("Outcome window: April 2026")
print("June 2026 remains sealed.")

Secure setup complete.
Feature window: March 2026
Outcome window: April 2026
June 2026 remains sealed.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features — calculated using March 2026 only

1. `march_impressions` — total measured Search Console impressions during March.
2. `march_clicks` — total measured Search Console clicks during March.
3. `march_ctr_pct` — March clicks divided by March impressions, expressed as a percentage.
4. `march_avg_position` — March average search position, weighted by impressions.
5. `march_active_search_days` — number of March dates on which the content item received at least one search impression.

These are the only five model features. Each is available by the decision moment at the end of 31 March 2026.

#### Label / proxy

- `declined_next_month` — equals 1 when April clicks are at least 20% below March clicks; otherwise it equals 0.
- The label is calculated only for content items observed with available Search Console data in both months and with at least one March click.
- This is a future-traffic proxy. It does not prove that a page needs updating or that an update would improve its performance.

#### Context and eligibility fields

- `client_hash_id` — retained for joining and client-separated evaluation.
- `content_hash_id` — retained for identifying and joining a content item.
- `report_date` — used to assign records to the March feature window or April outcome window.
- `gsc_data_available` — used as an eligibility flag. Only rows where this field `IS TRUE` enter the analysis.

These fields provide context or control data eligibility; they are not model features.

#### Excluded from the model

- `client_hash_id` and `content_hash_id` — pseudonymous identifiers retained as context but excluded from the feature matrix.
- `april_clicks` — a future outcome used only to construct the proxy label.
- `april_available_days` — future-period coverage information used only for inspection.
- `LEAKED_april_click_change_pct` — calculated from March and April clicks solely for the deliberate leakage experiment, then deleted.
- Any other April or future-derived value — unavailable at the March decision moment.
- Rows where `gsc_data_available` is not `TRUE` — unavailable measurements are excluded rather than interpreted as zero activity.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Verification query 1 of 3: prove the data grain
grain_query = f"""
WITH march_source AS (
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
),
daily_duplicates AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM march_source
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
),
monthly_slice AS (
    SELECT
        client_hash_id,
        content_hash_id
    FROM march_source
    GROUP BY 1, 2
),
monthly_duplicates AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM monthly_slice
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*) FROM march_source) AS source_daily_rows,
    (
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT
                report_date,
                client_hash_id,
                content_hash_id
            FROM march_source
        )
    ) AS distinct_daily_keys,
    (SELECT COUNT(*) FROM daily_duplicates) AS duplicate_daily_keys,
    (SELECT COUNT(*) FROM monthly_slice) AS monthly_analysis_rows,
    (SELECT COUNT(*) FROM monthly_duplicates) AS duplicate_monthly_keys
"""

grain_result = con.execute(grain_query).fetchdf()
grain_result

,source_daily_rows,distinct_daily_keys,duplicate_daily_keys,monthly_analysis_rows,duplicate_monthly_keys
0,3611061,3611061,0,176738,0


**Interpretation:** After filtering to rows where `gsc_data_available IS TRUE`, March 2026 contains 3,611,061 daily records. Each date–client–content combination is unique. Aggregating these records produces 176,738 client–content analysis rows, with no duplicate monthly keys. This supports the stated analysis grain of one content item within one client for the month.

In [11]:
# Verification query 2 of 3: verify the slice size and date span
slice_query = f"""
SELECT
    COUNT(*) AS available_daily_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(CAST(report_date AS DATE)) AS first_date,
    MAX(CAST(report_date AS DATE)) AS last_date,
    COUNT(DISTINCT CAST(report_date AS DATE)) AS observed_dates
FROM {MARCH}
WHERE gsc_data_available IS TRUE
"""

slice_result = con.execute(slice_query).fetchdf()
slice_result

,available_daily_rows,clients,content_items,first_date,last_date,observed_dates
0,3611061,47,176738,2026-03-01,2026-03-31,31


**Interpretation:** The available March slice contains 3,611,061 daily records representing 47 clients and 176,738 content items. Its dates run from 1 March through 31 March 2026, with all 31 calendar dates represented.

In [12]:
# Verification query 3 of 3: measure GSC data availability
availability_query = f"""
SELECT
    COUNT(*) AS total_daily_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_daily_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_daily_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS available_row_pct,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT client_hash_id) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_clients,
    COUNT(DISTINCT content_hash_id) AS total_content_items,
    COUNT(DISTINCT content_hash_id) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_content_items
FROM {MARCH}
"""

availability_result = con.execute(availability_query).fetchdf()
availability_result

,total_daily_rows,available_daily_rows,unavailable_daily_rows,available_row_pct,total_clients,available_clients,total_content_items,available_content_items
0,9841378,3611061,6230317,36.69,55,47,331437,176738


**Interpretation:** March contains 9,841,378 daily records in total. Filtering with `gsc_data_available IS TRUE` retains 3,611,061 records, or 36.69%. The filtered slice covers 47 of 55 clients and 176,738 of 331,437 content items. Therefore, unavailable history must not be treated as zero search activity.

### Five features and when they are available

1. **`march_impressions`** — knowable at the decision moment because it uses only impressions recorded during March, before the April outcome window.
2. **`march_clicks`** — knowable at the decision moment because it uses only clicks recorded during March.
3. **`march_ctr_pct`** — knowable at the decision moment because it is calculated only from March clicks and March impressions.
4. **`march_avg_position`** — knowable at the decision moment because it is an impression-weighted summary of search positions observed during March.
5. **`march_active_search_days`** — knowable at the decision moment because it counts March dates on which the content item received at least one search impression.

The decision moment is the end of 31 March 2026, before any April outcome is observed.

In [13]:
# Build the five-feature frame from March only
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS march_impressions,

    SUM(gsc_clicks) AS march_clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        ELSE NULL
    END AS march_ctr_pct,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_avg_position * gsc_impressions)
             / SUM(gsc_impressions)
        ELSE NULL
    END AS march_avg_position,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_impressions > 0
    ) AS march_active_search_days

FROM {MARCH}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

feature_frame = con.execute(feature_query).fetchdf()

print("Feature-frame rows:", len(feature_frame))
print("Feature count:", len(feature_frame.columns) - 2)
feature_frame.head(10)

Feature-frame rows: 176738
Feature count: 5


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_active_search_days
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,6.893301,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.535346,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.435680,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,3.871795,31
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,0.448430,10.538117,31
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,1.041667,5.864583,31
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,0.318471,10.057325,31
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,0.259437,5.127643,31
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,10.0,0.280820,9.273799,31


**Feature-frame result:** The frame contains 176,738 client–content rows and exactly five March features. The two hash columns are retained only as context for joining and group-aware splitting; they are not model features. No April information appears in this feature frame.

In [14]:
# Build the future outcome from April data
april_outcome_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS april_clicks,
    COUNT(DISTINCT report_date) AS april_available_days
FROM {APRIL}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
"""

april_outcomes = con.execute(april_outcome_query).fetchdf()

# Keep content items observed in both March and April.
analysis_frame = feature_frame.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one",
)

# The percentage-decline label requires a non-zero March denominator.
analysis_frame = analysis_frame.loc[
    analysis_frame["march_clicks"] >= 1
].copy()

# Future proxy: April clicks are at least 20% below March clicks.
analysis_frame["declined_next_month"] = (
    analysis_frame["april_clicks"]
    <= 0.80 * analysis_frame["march_clicks"]
).astype("int8")

print("April content rows with available GSC data:", len(april_outcomes))
print("Eligible rows observed in both months with March clicks >= 1:",
      len(analysis_frame))
print()
print("Proxy-label counts:")
print(analysis_frame["declined_next_month"].value_counts().sort_index())
print()
print(
    "Declining rate:",
    f'{analysis_frame["declined_next_month"].mean():.3f}'
)

analysis_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "march_clicks",
        "april_clicks",
        "april_available_days",
        "declined_next_month",
    ]
].head(10)

April content rows with available GSC data: 194760
Eligible rows observed in both months with March clicks >= 1: 68040

Proxy-label counts:
declined_next_month
0    26538
1    41502
Name: count, dtype: int64

Declining rate: 0.610


,client_hash_id,content_hash_id,march_clicks,april_clicks,april_available_days,declined_next_month
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,8.0,30,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,4.0,30,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,8.0,30,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,0.0,30,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,1.0,2.0,30,0
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,1.0,2.0,29,0
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,1.0,0.0,30,1
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,20.0,15.0,30,1
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,10.0,12.0,30,0
10,client_73cda7b4e4f265ea,content_f71459b346aba398,1.0,0.0,27,1


**Proxy-label result:** Of the content items observed in both months, 68,040 had at least one March click and were eligible for the percentage comparison. The proxy marks 41,502 items (61.0%) as declining. This is a future-traffic proxy, not proof that an editor should refresh the page or that refreshing it would improve performance.

In [15]:
# Deliberate leakage experiment: honest model versus one leaked feature
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

honest_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_search_days",
]

target = "declined_next_month"
groups = analysis_frame["client_hash_id"]

# Keep entire clients together during evaluation.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        analysis_frame[honest_features],
        analysis_frame[target],
        groups=groups,
    )
)

train_clients = set(analysis_frame.iloc[train_idx]["client_hash_id"])
test_clients = set(analysis_frame.iloc[test_idx]["client_hash_id"])

assert train_clients.isdisjoint(test_clients)

def make_quick_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=42,
        ),
    )

# Honest model: March information only
honest_model = make_quick_model()
honest_model.fit(
    analysis_frame.iloc[train_idx][honest_features],
    analysis_frame.iloc[train_idx][target],
)

honest_scores = honest_model.predict_proba(
    analysis_frame.iloc[test_idx][honest_features]
)[:, 1]

honest_auc = roc_auc_score(
    analysis_frame.iloc[test_idx][target],
    honest_scores,
)

# Add exactly one deliberately leaked feature.
leak_column = "LEAKED_april_click_change_pct"

analysis_frame[leak_column] = (
    100.0
    * (
        analysis_frame["april_clicks"]
        - analysis_frame["march_clicks"]
    )
    / analysis_frame["march_clicks"]
)

leaky_features = honest_features + [leak_column]

leaky_model = make_quick_model()
leaky_model.fit(
    analysis_frame.iloc[train_idx][leaky_features],
    analysis_frame.iloc[train_idx][target],
)

leaky_scores = leaky_model.predict_proba(
    analysis_frame.iloc[test_idx][leaky_features]
)[:, 1]

leaky_auc = roc_auc_score(
    analysis_frame.iloc[test_idx][target],
    leaky_scores,
)

print("Training clients:", len(train_clients))
print("Held-out clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Training rows:", len(train_idx))
print("Held-out rows:", len(test_idx))
print()
print(f"Honest March-only ROC-AUC: {honest_auc:.3f}")
print(f"Leaky ROC-AUC:             {leaky_auc:.3f}")
print(f"Score increase:            {leaky_auc - honest_auc:.3f}")

# Remove the prohibited future-derived feature immediately.
analysis_frame.drop(columns=[leak_column], inplace=True)

print()
print("Leaked column deleted:", leak_column not in analysis_frame.columns)
print("Honest feature count retained:", len(honest_features))

Training clients: 33
Held-out clients: 9
Client overlap: 0
Training rows: 64544
Held-out rows: 3496

Honest March-only ROC-AUC: 0.653
Leaky ROC-AUC:             1.000
Score increase:            0.347

Leaked column deleted: True
Honest feature count retained: 5


**Leakage result:** The honest model achieved a held-out ROC-AUC of 0.653 using only the five March features. Adding `LEAKED_april_click_change_pct` raised the score to 1.000 because that column uses April clicks and directly reproduces the rule used to create the label. It would not be known at the March decision moment. I deleted it immediately and retained only the five honest features. Clients were kept separate between training and evaluation.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation: uneven GSC coverage

Only 36.69% of March daily records had `gsc_data_available IS TRUE`. The available slice represented 47 of 55 clients and 176,738 of 331,437 content items. Therefore, this analysis reflects only clients and content with available Search Console measurements and may not represent the excluded portion of the warehouse. Missing or unavailable history must not be interpreted as zero search activity.

In [16]:
# Final reproducibility and requirement checks
expected_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_search_days",
]

assert len(expected_features) == 5
assert all(column in feature_frame.columns for column in expected_features)

assert feature_frame.duplicated(
    ["client_hash_id", "content_hash_id"]
).sum() == 0

assert int(grain_result.loc[0, "duplicate_daily_keys"]) == 0
assert int(grain_result.loc[0, "duplicate_monthly_keys"]) == 0

assert (
    int(slice_result.loc[0, "available_daily_rows"])
    == int(grain_result.loc[0, "source_daily_rows"])
)

assert int(slice_result.loc[0, "observed_dates"]) == 31
assert analysis_frame["declined_next_month"].isin([0, 1]).all()
assert leak_column not in analysis_frame.columns
assert train_clients.isdisjoint(test_clients)

print("All self-checks passed.")
print("Verification queries completed: 3")
print("Honest features retained: 5")
print("Leaked feature present: False")
print("June 2026 used for development: False")

All self-checks passed.
Verification queries completed: 3
Honest features retained: 5
Leaked feature present: False
June 2026 used for development: False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.